In [9]:
import requests
from bs4 import BeautifulSoup
import time
import re
import json
from datetime import datetime
from collections import defaultdict

SIGNALS = {
    "Не хватает времени": ["успевать", "готовить времени нет", "быстрый обед", "перекус на ходу", "нет времени", "быстро поесть", "на бегу"],
    "Следит за фигурой": ["калории", "белок", "бжу", "жирно", "диета", "пп", "правильное питание", "зож", "похудеть", "фигура"],
    "Работает в офисе": ["офис", "кулер", "доставка в офис", "коллега", "обеденный перерыв", "работа", "бизнес-ланч", "офисный"],
    "Белок/спорт": ["протеин", "курица", "гречка", "рис", "творог", "тренировка", "фитнес", "зал", "мышцы"],
    "Экономия": ["дорого", "дешево", "цена", "бюджет", "экономить", "скидка", "акция"],
    "Фастфуд": ["бургер", "пицца", "картошка фри", "фастфуд", "макдональдс", "кфс"]
}

def parse_pikabu_posts(url, max_posts=30):
    """Парсит посты с Пикабу и собирает информацию"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'ru-RU,ru;q=0.9,en;q=0.8',
    }

    collected_posts = []

    try:
        print(f"Загружаем страницу: {url}")
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')
        posts_found = []

        posts_found = soup.find_all('article', class_=re.compile('story|post'))

        if not posts_found:
            posts_found = soup.find_all('div', class_=re.compile('story__|post__'))

        if not posts_found:
            posts_found = soup.find_all(['article', 'div'], class_=True)
            posts_found = [p for p in posts_found if p.find(['h1', 'h2', 'h3'])]

        print(f"Найдено {len(posts_found)} потенциальных постов")

        for idx, post in enumerate(posts_found[:max_posts]):
            try:
                title_tag = post.find(['h1', 'h2', 'h3', 'a', 'span'],
                                     class_=re.compile('title|header|heading'))
                if not title_tag:
                    title_tag = post.find(['a', 'span'], string=re.compile(r'.{10,}'))

                title = title_tag.text.strip() if title_tag else ""
                if len(title) < 10:
                    continue

                content_tag = post.find(['div', 'p', 'span'],
                                       class_=re.compile('content|text|body|description'))
                content = content_tag.text.strip() if content_tag else title

                likes = "0"
                likes_tag = post.find(class_=re.compile('likes|rating|votes'))
                if likes_tag:
                    likes_raw = likes_tag.text.strip()
                    likes = re.search(r'\d+', likes_raw).group() if re.search(r'\d+', likes_raw) else "0"

                date_tag = post.find(['time', 'span'], class_=re.compile('date|time'))
                date = date_tag.text.strip() if date_tag else datetime.now().strftime("%Y-%m-%d")

                link_tag = post.find('a', href=re.compile(r'/story/|/post/'))
                link = link_tag['href'] if link_tag else ""
                if link and not link.startswith('http'):
                    link = "https://pikabu.ru" + link

                full_text = f"{title} {content}".lower()
                matched_signals = []

                for category, keywords in SIGNALS.items():
                    if any(keyword in full_text for keyword in keywords):
                        matched_signals.append(category)

                post_data = {
                    'title': title,
                    'content_preview': content[:200] + "...",
                    'likes': likes,
                    'date': date,
                    'url': link,
                    'signals': ", ".join(matched_signals) if matched_signals else "Нет сигналов",
                    'signal_count': len(matched_signals)
                }
                collected_posts.append(post_data)

                if matched_signals:
                    print(f"\nПОСТ {idx+1}:")
                    print(f"   Заголовок: {title[:60]}...")
                    print(f"   Сигналы: {', '.join(matched_signals)}")
                    print(f"   Лайков: {likes}")

            except Exception as e:
                print(f"Ошибка при обработке поста {idx}: {e}")
                continue

            time.sleep(0.5)

    except requests.RequestException as e:
        print(f"Ошибка загрузки страницы: {e}")
    except Exception as e:
        print(f"Непредвиденная ошибка: {e}")

    return collected_posts

def analyze_collected_data(posts):
    """Анализирует собранные данные и выдает статистику"""
    if not posts:
        print("Нет данных для анализа")
        return

    print("\n")
    print("АНАЛИЗ ЦЕЛЕВОЙ АУДИТОРИИ ПО ФОРУМАМ")

    signal_counter = defaultdict(int)
    for post in posts:
        if post['signals'] != "Нет сигналов":
            signals = post['signals'].split(", ")
            for signal in signals:
                signal_counter[signal] += 1

    print("\nТОП-5 БОЛЕЙ И ПОТРЕБНОСТЕЙ АУДИТОРИИ:")
    sorted_signals = sorted(signal_counter.items(), key=lambda x: x[1], reverse=True)
    for signal, count in sorted_signals[:5]:
        percentage = (count / len(posts)) * 100
        print(f"{signal}: {count} упоминаний ({percentage:.1f}%)")

    print("\nСАМЫЕ РЕЛЕВАНТНЫЕ ПОСТЫ (несколько сигналов сразу):")
    top_posts = sorted(posts, key=lambda x: x['signal_count'], reverse=True)[:3]
    for post in top_posts:
        if post['signal_count'] > 0:
            print(f"\n{post['title'][:70]}...")
            print(f" Сигналы: {post['signals']}")
            print(f" Лайков: {post['likes']}")
            if post['url']:
                print(f" Ссылка: {post['url']}")

    total_likes = sum(int(post['likes']) for post in posts if post['likes'].isdigit())
    avg_likes = total_likes // len(posts) if posts else 0

    print("\nОБЩАЯ СТАТИСТИКА:")
    print(f"   Всего проанализировано постов: {len(posts)}")
    print(f"   Среднее количество лайков: {avg_likes}")
    print(f"   Посты с сигналами: {sum(1 for p in posts if p['signal_count'] > 0)} из {len(posts)}")

    filename = f"pikabu_analysis_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(posts, f, ensure_ascii=False, indent=2)
    print(f"\nДанные сохранены в файл: {filename}")
    try:
        import csv
        csv_filename = filename.replace('.json', '.csv')
        with open(csv_filename, 'w', encoding='utf-8-sig', newline='') as f:
            fieldnames = ['title', 'likes', 'date', 'signals', 'signal_count', 'url']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for post in posts:
                row = {key: post.get(key, '') for key in fieldnames}
                writer.writerow(row)
    except Exception as e:
        print(f"Ошибка при сохранении CSV: {e}")

def parse_multiple_pages():
    """Парсит несколько страниц с разными тегами"""
    topics = [
        ("Обед", "https://pikabu.ru/tag/Обед/hot"),
        ("ПП", "https://pikabu.ru/tag/ПП/hot"),
        ("Фитнес", "https://pikabu.ru/tag/Фитнес/hot"),
        ("Доставка еды", "https://pikabu.ru/tag/Доставка%20еды/hot"),
        ("ЗОЖ", "https://pikabu.ru/tag/ЗОЖ/hot"),
        ("Еда", "https://pikabu.ru/tag/Еда/hot"),
    ]
    all_posts = []

    for topic_name, url in topics:
        print(f"\n")
        print(f"Парсинг темы: {topic_name}")
        posts = parse_pikabu_posts(url, max_posts=15)
        all_posts.extend(posts)
        print(f"Собрано {len(posts)} постов по теме '{topic_name}'")
        time.sleep(2)

    print(f"\n")
    print(f"ИТОГО СОБРАНО: {len(all_posts)} ПОСТОВ")
    analyze_collected_data(all_posts)

    return all_posts

if __name__ == "__main__":
    print("ЗАПУСК ПАРСИНГА ФОРУМОВ ДЛЯ АНАЛИЗА ЦА САЛАТ-БАРА")

    all_posts = parse_multiple_pages()

ЗАПУСК ПАРСИНГА ФОРУМОВ ДЛЯ АНАЛИЗА ЦА САЛАТ-БАРА


Парсинг темы: Обед
Загружаем страницу: https://pikabu.ru/tag/Обед/hot
Найдено 10 потенциальных постов

ПОСТ 1:
   Заголовок: Если мучное манит, а вес не уходит⁠⁠...
   Сигналы: Следит за фигурой, Белок/спорт
   Лайков: 0

ПОСТ 2:
   Заголовок: Курица Гунбао. Обзор китайских инноваций в сфере доширака⁠⁠...
   Сигналы: Белок/спорт, Фастфуд
   Лайков: 4

ПОСТ 3:
   Заголовок: Мужская кухня. Сливочный суп с кукурузой и курицей⁠⁠...
   Сигналы: Следит за фигурой, Белок/спорт
   Лайков: 13

ПОСТ 5:
   Заголовок: Обзор на автомобильный чайник, с подогревом от прикуривателя...
   Сигналы: Работает в офисе, Белок/спорт, Экономия
   Лайков: 2

ПОСТ 6:
   Заголовок: Обед с бабушкой в лесу⁠⁠...
   Сигналы: Белок/спорт
   Лайков: 12

ПОСТ 8:
   Заголовок: Традиционная уличная еда Пакистана. Фастфуд⁠⁠...
   Сигналы: Фастфуд
   Лайков: 3
Собрано 8 постов по теме 'Обед'


Парсинг темы: ПП
Загружаем страницу: https://pikabu.ru/tag/ПП/hot
Найдено 10 по

In [10]:
import requests
from bs4 import BeautifulSoup
import time
import re
import json
from datetime import datetime
from collections import defaultdict

SIGNALS = {
    "Не хватает времени": ["нет времени", "быстрый", "готовить некогда", "экономия времени", "не надо готовить", "освободилось время"],
    "Следит за фигурой/ПП": ["пп", "зож", "калории", "бжу", "диета", "фигура", "похудеть", "дефицит калорий", "правильное питание"],
    "Работает в офисе": ["офис", "работа", "коллеги", "бизнес-ланч", "обед на работе", "офисный работник"],
    "Фитнес/спорт": ["фитнес", "тренировка", "зал", "белок", "протеин", "мышцы", "спорт"],
    "Проблемы с доставкой": ["доставка", "курьер", "опоздали", "время доставки", "задержалась"],
    "Качество еды": ["вкусно", "невкусно", "свежий", "прокис", "качество", "порции"],
    "Цена/бюджет": ["дорого", "дешево", "бюджет", "цена", "экономия", "тысяч"]
}

def parse_irecommend_category(category_url, max_reviews=20):
    """
    Парсит отзывы из категории IRecommend
    Использует прямые URL категорий вместо поиска
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    collected_reviews = []

    try:
        print(f"Загружаем: {category_url}")
        response = requests.get(category_url, headers=headers, timeout=15)
        soup = BeautifulSoup(response.text, 'html.parser')

        review_links = []

        for link in soup.find_all('a', href=True):
            href = link['href']
            if '/content/' in href and href not in review_links:
                if href.startswith('/content/'):
                    full_link = "https://irecommend.ru" + href
                    review_links.append(full_link)

        review_blocks = soup.find_all('div', class_=re.compile('review|content-item|product-review'))
        for block in review_blocks:
            link = block.find('a', href=re.compile(r'/content/'))
            if link and link['href']:
                full_link = "https://irecommend.ru" + link['href']
                if full_link not in review_links:
                    review_links.append(full_link)

        print(f"Найдено ссылок: {len(review_links)}")

        review_links = review_links[:max_reviews]

        for idx, link in enumerate(review_links):
            try:
                time.sleep(1.5)

                review_response = requests.get(link, headers=headers, timeout=15)
                review_soup = BeautifulSoup(review_response.text, 'html.parser')

                title_tag = review_soup.find('h1')
                title = title_tag.text.strip() if title_tag else "Без заголовка"

                content = ""
                content_selectors = [
                    ('div', {'class': re.compile(r'field-type-text-with-summary|content|body')}),
                    ('div', {'itemprop': 'reviewBody'}),
                    ('div', {'class': 'content clearfix'})
                ]

                for tag, attrs in content_selectors:
                    content_tag = review_soup.find(tag, attrs)
                    if content_tag:
                        content = content_tag.text.strip()
                        break

                if not content:
                    for div in review_soup.find_all('div'):
                        if len(div.text) > 200 and 'отзыв' not in div.text.lower():
                            content = div.text.strip()
                            break

                author_tag = review_soup.find('span', class_=re.compile('username|author'))
                if not author_tag:
                    author_tag = review_soup.find('a', class_=re.compile('user'))
                author = author_tag.text.strip() if author_tag else "Аноним"

                date_tag = review_soup.find('span', class_=re.compile('submitted|date|time'))
                if not date_tag:
                    date_tag = review_soup.find('time')
                date = date_tag.text.strip() if date_tag else ""

                rating = 0
                rating_tag = review_soup.find('span', class_=re.compile('rating|stars'))
                if rating_tag:
                    stars = rating_tag.find_all('span', class_=re.compile('star'))
                    rating = sum(1 for star in stars if 'full' in str(star) or 'active' in str(star))

                full_text = f"{title} {content[:1000]}".lower()
                matched_signals = []

                for category, keywords in SIGNALS.items():
                    for keyword in keywords:
                        if keyword.lower() in full_text:
                            matched_signals.append(category)
                            break
                matched_signals = list(set(matched_signals))
                review_data = {
                    'source': 'IRecommend',
                    'url': link,
                    'title': title[:100],
                    'content': content[:700] + "..." if len(content) > 700 else content,
                    'author': author,
                    'date': date,
                    'rating': rating,
                    'signals': ", ".join(matched_signals) if matched_signals else "Нет сигналов",
                    'signal_count': len(matched_signals)
                }
                collected_reviews.append(review_data)

                if matched_signals:
                    print(f"\n[{idx+1}] {title[:50]}...")
                    print(f"Сигналы: {', '.join(matched_signals)}")
                    print(f"Рейтинг: {rating}/5 звезд")
                else:
                    print(f"[{idx+1}] {title[:40]}... (сигналов нет)")

            except Exception as e:
                print(f"Ошибка: {e}")
                continue

        return collected_reviews
    except Exception as e:
        print(f"Ошибка: {e}")
        return []

def parse_specific_reviews():
    specific_urls = [
        "https://irecommend.ru/content/pravilnoe-pitanie-s-dostavkoi-na-dom-ot-daily-food-umerennyi-defitsit-kalorii-i-sbros-lishne",
        "https://irecommend.ru/content/ochen-udobno-404",
        "https://irecommend.ru/content/blestyashche-72",
        "https://irecommend.ru/content/vkusno-1881",
        "https://irecommend.ru/content/poprobovat-edu-tak-i-ne-udalos",
        "https://irecommend.ru/content/ne-rekomenduyu-2572",
        "https://irecommend.ru/content/uzhasnoe-kachestvo-i-obman-0",
    ]
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    collected_reviews = []

    print("Парсинг конкретных отзывов о доставках еды")
    for idx, url in enumerate(specific_urls):
        try:
            time.sleep(1.5)
            response = requests.get(url, headers=headers, timeout=15)
            soup = BeautifulSoup(response.text, 'html.parser')
            title_tag = soup.find('h1')
            title = title_tag.text.strip() if title_tag else "Без заголовка"

            content = ""
            content_selectors = [
                'div[itemprop="reviewBody"]',
                'div.field-type-text-with-summary',
                'div.content',
                'div.clearfix'
            ]

            for selector in content_selectors:
                content_tag = soup.select_one(selector)
                if content_tag:
                    content = content_tag.text.strip()
                    break
            if not content:
                for div in soup.find_all('div'):
                    if len(div.text) > 300:
                        content = div.text.strip()
                        break

            author_tag = soup.find('span', class_=re.compile('username|author'))
            author = author_tag.text.strip() if author_tag else "Аноним"
            date_tag = soup.find('span', class_=re.compile('submitted|date'))
            date = date_tag.text.strip() if date_tag else ""

            rating = 0
            rating_stars = soup.find_all('span', class_='star')
            rating = sum(1 for star in rating_stars if 'full' in str(star) or 'active' in str(star))

            full_text = f"{title} {content}".lower()
            matched_signals = []

            for category, keywords in SIGNALS.items():
                for keyword in keywords:
                    if keyword.lower() in full_text:
                        matched_signals.append(category)
                        break

            matched_signals = list(set(matched_signals))

            review_data = {
                'source': 'IRecommend (direct)',
                'url': url,
                'title': title[:100],
                'content': content[:800] + "..." if len(content) > 800 else content,
                'author': author,
                'date': date,
                'rating': rating,
                'signals': ", ".join(matched_signals) if matched_signals else "Нет сигналов",
                'signal_count': len(matched_signals)
            }

            collected_reviews.append(review_data)
        except Exception as e:
            print(f"Ошибка: {e}")
            continue
    return collected_reviews

def analyze_reviews(reviews):
    if not reviews:
        print("Нет данных для анализа")
        return

    signal_counter = defaultdict(int)
    for review in reviews:
        if review['signals'] != "Нет сигналов":
            for signal in review['signals'].split(", "):
                signal_counter[signal] += 1

    print("РЕЗУЛЬТАТЫ АНАЛИЗА ОТЗЫВОВ")

    print("\nТОП ПОТРЕБНОСТЕЙ АУДИТОРИИ:")
    for signal, count in sorted(signal_counter.items(), key=lambda x: x[1], reverse=True):
        percentage = (count / len(reviews)) * 100
        print(f"   - {signal}: {count} упоминаний ({percentage:.1f}%)")

    print("\nКлючевые выводы")

    if signal_counter.get("Не хватает времени", 0) > 0:
        print("Люди ценят экономию времени")
    if signal_counter.get("Следит за фигурой/ПП", 0) > 0:
        print("Аудитория следит за питанием")
    if signal_counter.get("Фитнес/спорт", 0) > 0:
        print("Есть запрос на спортивное питание")
    if signal_counter.get("Работает в офисе", 0) > 0:
        print("Спрос со стороны офисных сотрудники")
    if signal_counter.get("Цена/бюджет", 0) > 0:
        print("Цена важна для потребителей, нужны бюджетные тарифы или акции")

    filename = f"irecommend_analysis_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(reviews, f, ensure_ascii=False, indent=2)
    print(f"\nСохранено в: {filename}")

print("ПАРСИНГ ОТЗЫВОВ IRecommend ДЛЯ АНАЛИЗА ЦА")

all_reviews = []
direct_reviews = parse_specific_reviews()
all_reviews.extend(direct_reviews)

print(f"\nВсего собрано отзывов: {len(all_reviews)}")
analyze_reviews(all_reviews)

ПАРСИНГ ОТЗЫВОВ IRecommend ДЛЯ АНАЛИЗА ЦА
Парсинг конкретных отзывов о доставках еды

Всего собрано отзывов: 7
РЕЗУЛЬТАТЫ АНАЛИЗА ОТЗЫВОВ

ТОП ПОТРЕБНОСТЕЙ АУДИТОРИИ:

Ключевые выводы

Сохранено в: irecommend_analysis_20260525_1449.json


In [11]:
!apt-get update
!apt-get install -y wget unzip xvfb libxi6 libgconf-2-4
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google.list
!apt-get update
!apt-get install -y google-chrome-stable

Hit:1 http://dl.google.com/linux/chrome/deb stable InRelease
Get:2 https://dl.google.com/linux/chrome-stable/deb stable InRelease [1,825 B]
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:5 https://dl.google.com/linux/chrome-stable/deb stable/main amd64 Packages [1,220 B]
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,045 B in 1s (2,168 B/s)
Reading package lists... Done
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc

In [12]:
!apt-get install -y -qq chromium-browser chromium-chromedriver

W: Target Packages (main/binary-amd64/Packages) is configured multiple times in /etc/apt/sources.list.d/google.list:1 and /etc/apt/sources.list.d/google.list:2
W: Target Packages (main/binary-all/Packages) is configured multiple times in /etc/apt/sources.list.d/google.list:1 and /etc/apt/sources.list.d/google.list:2


In [13]:
!pip install selenium -q

In [14]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import json
from datetime import datetime
from collections import defaultdict

options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

SIGNALS = {
    "Не хватает времени": [
        "нет времени", "быстрый", "готовить некогда", "быстро поесть",
        "перекус", "цейтнот", "быстро", "некогда", "экономия времени"
    ],
    "Следит за фигурой/ПП": [
        "пп", "зож", "калории", "бжу", "диета", "фигура", "похудеть",
        "правильное питание", "полезно", "диетическое", "лёгкий", "кбжу"
    ],
    "Работает в офисе": [
        "офис", "работа", "коллеги", "бизнес-ланч", "обед на работе",
        "бизнес-центр", "офисный", "доставка в офис"
    ],
    "Фитнес/спорт": [
        "фитнес", "тренировка", "зал", "белок", "протеин", "спорт",
        "тренируюсь", "после тренировки", "спортзал"
    ],
    "Качество еды": [
        "вкусно", "невкусно", "свежий", "прокис", "качество", "порции",
        "холодный", "вкус", "безвкусно", "свежесть"
    ],
    "Цена/бюджет": [
        "дорого", "дешево", "цена", "экономия", "скидка", "акция",
        "бюджетно", "ценник", "стоимость"
    ],
    "Проблемы с сервисом": [
        "обслуживание", "персонал", "хам", "долго", "очередь",
        "медленно", "не убрали", "грязно"
    ]
}

search_queries = [
    "фитнес клуб",
    "салат бар",
    "здоровое питание",
    "пп кафе",
    "бизнес ланч"
]

def search_and_parse_flamp(query, max_places=5, max_reviews_per_place=15):
    all_reviews = []
    search_url = f"https://flamp.ru/search/{query.replace(' ', '%20')}"
    print(f"\n Поиск: {query}")
    print(f"   URL: {search_url}")

    try:
        driver.get(search_url)
        time.sleep(5)

        for _ in range(2):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
        place_links = []
        selectors = [
            "a[href*='/firm/']",
            "div[class*='item'] a",
            "a[class*='title']"
        ]
        for selector in selectors:
            links = driver.find_elements(By.CSS_SELECTOR, selector)
            for link in links:
                href = link.get_attribute('href')
                if href and '/firm/' in href and href not in place_links:
                    place_links.append(href)
            if place_links:
                print(f"Найдено ссылок: {len(place_links)}")
                break

        if not place_links:
            print("Организации не найдены")
            return all_reviews

        place_links = place_links[:max_places]
        for idx, firm_url in enumerate(place_links):
            try:
                print(f"\n[{idx+1}/{len(place_links)}] Загружаем: {firm_url}")
                driver.get(firm_url)
                time.sleep(4)

                for scroll in range(5):
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                    time.sleep(2)
                    print(f"Прокрутка {scroll+1}/5", end="\r")

                print()
                review_elements = []
                review_selectors = [
                    "div[class*='review']",
                    "div[class*='Review']",
                    "div[data-testid='review-card']"
                ]

                for selector in review_selectors:
                    review_elements = driver.find_elements(By.CSS_SELECTOR, selector)
                    if review_elements:
                        print(f"Найдено отзывов: {len(review_elements)}")
                        break

                if not review_elements:
                    print(f"Отзывы не найдены")
                    continue

                org_name = ""
                name_elem = driver.find_elements(By.CSS_SELECTOR, "h1, div[class*='name']")
                if name_elem:
                    org_name = name_elem[0].text.strip()[:50]
                review_elements = review_elements[:max_reviews_per_place]

                for rev_idx, elem in enumerate(review_elements):
                    try:
                        text = elem.text.strip()

                        if len(text) < 30:
                            continue
                        author = "Аноним"
                        author_elem = elem.find_elements(By.CSS_SELECTOR, "span[class*='name'], div[class*='author']")
                        if author_elem:
                            author = author_elem[0].text.strip()[:40]
                        rating = 0
                        rating_elem = elem.find_elements(By.CSS_SELECTOR, "span[class*='star'], div[class*='rating']")
                        for r in rating_elem:
                            if 'active' in r.get_attribute('class') or 'filled' in r.get_attribute('class'):
                                rating += 1

                        text_lower = text.lower()
                        matched_signals = []

                        for category, keywords in SIGNALS.items():
                            for keyword in keywords:
                                if keyword in text_lower:
                                    matched_signals.append(category)
                                    break

                        matched_signals = list(set(matched_signals))
                        if matched_signals:
                            review_data = {
                                "query": query,
                                "organization": org_name,
                                "author": author,
                                "rating": rating,
                                "text": text[:500],
                                "signals": ", ".join(matched_signals)
                            }
                            all_reviews.append(review_data)

                            star_display = rating if rating > 0 else "?"
                            print(f"\n[{rev_idx+1}] {author} {star_display}")
                            print(f"{text[:100]}...")
                            print(f"{', '.join(matched_signals)}")

                    except Exception as e:
                        continue

                time.sleep(2)

            except Exception as e:
                print(f"Ошибка при загрузке: {e}")
                continue

    except Exception as e:
        print(f"Ошибка поиска: {e}")

    return all_reviews

print("FLAMP.RU")
print(f"\nПоисковые запросы: {', '.join(search_queries)}")
print(f"Целевая аудитория: люди, которые ходят в фитнес и следят за питанием")

all_reviews = []

for query in search_queries:
    reviews = search_and_parse_flamp(query, max_places=4, max_reviews_per_place=15)
    all_reviews.extend(reviews)
    print(f"\nИтого по запросу '{query}': {len(reviews)} отзывов с сигналами")
    time.sleep(3)

driver.quit()

print("ОБЩИЙ АНАЛИЗ СОБРАННЫХ ДАННЫХ")


if not all_reviews:
    print(" Отзывы не найдены")
    print("\nВозможные причины:")
    print("1. Сайт временно недоступен")
    print("2. Нет отзывов по этим запросам")
else:
    signal_counter = defaultdict(int)
    for review in all_reviews:
        for signal in review["signals"].split(", "):
            signal_counter[signal] += 1

    print(f"\nСтатистика:")
    print(f"Всего собрано отзывов: {len(all_reviews)}")
    print(f"Уникальных организаций: {len(set(r['organization'] for r in all_reviews))}")

    print(f"\nТОП ПОТРЕБНОСТЕЙ И ПРОБЛЕМ АУДИТОРИИ:")
    sorted_signals = sorted(signal_counter.items(), key=lambda x: x[1], reverse=True)
    for signal, count in sorted_signals:
        percentage = (count / len(all_reviews)) * 100
        bar_length = int(percentage / 2)
        bar = "█" * bar_length + "░" * (20 - bar_length)
        print(f"   {signal:22} | {bar} {count} ({percentage:.0f}%)")

    print(f"\nКЛЮЧЕВЫЕ ВЫВОДЫ:")

    insights_list = []
    if signal_counter.get("Не хватает времени", 0) > 0:
        insights_list.append("Делаем акцент на быстроте")
    if signal_counter.get("Следит за фигурой/ПП", 0) > 0:
        insights_list.append("Указываем КБЖУ на каждом блюде")
    if signal_counter.get("Фитнес/спорт", 0) > 0:
        insights_list.append("Добавляем высокобелковые опции")
    if signal_counter.get("Работает в офисе", 0) > 0:
        insights_list.append("Предлагаем бизнес-ланчи")
    if signal_counter.get("Качество еды", 0) > 0:
        insights_list.append("Делаем упор на свежесть ингридиентов")
    if signal_counter.get("Цена/бюджет", 0) > 0:
        insights_list.append("Добавляем бюджетные позиции и программу лояльности")
    if signal_counter.get("Проблемы с сервисом", 0) > 0:
        insights_list.append("Обучаем персонал вежливости, добавляем кассы самообмлуживания")

    for insight in insights_list:
        print(f"{insight}")

    if not insights_list:
        print("Соберите больше отзывов для точных выводов")

    print(f"\nПРИМЕРЫ ОТЗЫВОВ:")
    signal_reviews = [r for r in all_reviews if r['signals'] != "Нет сигналов"]
    for i, review in enumerate(signal_reviews[:5], 1):
        print(f"\n   {i}. {review['organization']}")
        print(f"   {review['author']} {review['rating']}")
        print(f"   {review['text'][:150]}...")
        print(f"   Сигналы: {review['signals']}")

    filename = f"flamp_analysis_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(all_reviews, f, ensure_ascii=False, indent=2)
    print(f"\nСохранено в: {filename}")

    txt_filename = filename.replace('.json', '_report.txt')
    with open(txt_filename, 'w', encoding='utf-8') as f:
        f.write("ОТЧЕТ ПО АНАЛИЗУ ОТЗЫВОВ FLAMP.RU\n")
        f.write(f"Всего отзывов: {len(all_reviews)}\n")
        f.write(f"Отзывов с сигналами: {len(signal_reviews)}\n\n")
        f.write("ТОП ПОТРЕБНОСТЕЙ:\n")
        for signal, count in sorted_signals:
            f.write(f"  {signal}: {count}\n")
        f.write("\n")
        f.write("ПРИМЕРЫ ОТЗЫВОВ:\n\n")
        for review in signal_reviews[:10]:
            f.write(f"Организация: {review['organization']}\n")
            f.write(f"Автор: {review['author']}\n")
            f.write(f"Рейтинг: {review['rating']}\n")
            f.write(f"Сигналы: {review['signals']}\n")
            f.write(f"Текст: {review['text']}\n")
            f.write("\n\n")

    print(f"Текстовый отчет: {txt_filename}")

print(f"\nВсего обработано отзывов: {len(all_reviews)}")

FLAMP.RU

Поисковые запросы: фитнес клуб, салат бар, здоровое питание, пп кафе, бизнес ланч
Целевая аудитория: люди, которые ходят в фитнес и следят за питанием

 Поиск: фитнес клуб
   URL: https://flamp.ru/search/фитнес%20клуб
Организации не найдены

Итого по запросу 'фитнес клуб': 0 отзывов с сигналами

 Поиск: салат бар
   URL: https://flamp.ru/search/салат%20бар
Организации не найдены

Итого по запросу 'салат бар': 0 отзывов с сигналами

 Поиск: здоровое питание
   URL: https://flamp.ru/search/здоровое%20питание
Организации не найдены

Итого по запросу 'здоровое питание': 0 отзывов с сигналами

 Поиск: пп кафе
   URL: https://flamp.ru/search/пп%20кафе
Организации не найдены

Итого по запросу 'пп кафе': 0 отзывов с сигналами

 Поиск: бизнес ланч
   URL: https://flamp.ru/search/бизнес%20ланч
Организации не найдены

Итого по запросу 'бизнес ланч': 0 отзывов с сигналами
ОБЩИЙ АНАЛИЗ СОБРАННЫХ ДАННЫХ
 Отзывы не найдены

Возможные причины:
1. Сайт временно недоступен
2. Нет отзывов по эт

In [17]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import json
from datetime import datetime
from collections import defaultdict

options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

SIGNALS = {
    "Не хватает времени": [
        "нет времени", "быстрый", "готовить некогда", "быстро поесть",
        "перекус", "цейтнот", "быстро", "некогда", "экономия времени",
        "оперативно", "быстрое обслуживание", "не ждать"
    ],
    "Следит за фигурой/ПП": [
        "пп", "зож", "калории", "бжу", "диета", "фигура", "похудеть",
        "правильное питание", "полезно", "диетическое", "лёгкий", "кбжу",
        "здоровое питание", "спортивное питание", "фигура"
    ],
    "Работает в офисе": [
        "офис", "работа", "коллеги", "бизнес-ланч", "обед на работе",
        "бизнес-центр", "офисный", "доставка в офис", "в обед",
        "обеденное время", "перерыв"
    ],
    "Фитнес/спорт": [
        "фитнес", "тренировка", "зал", "белок", "протеин", "спорт",
        "тренируюсь", "после тренировки", "спортзал", "тренажерный",
        "силовые", "кардио", "персональный тренер"
    ],
    "Качество еды": [
        "вкусно", "невкусно", "свежий", "прокис", "качество", "порции",
        "холодный", "вкус", "безвкусно", "свежесть", "тухлый",
        "просрочка", "невкусный"
    ],
    "Цена/бюджет": [
        "дорого", "дешево", "цена", "экономия", "скидка", "акция",
        "бюджетно", "ценник", "стоимость", "завышенные цены",
        "неоправданно дорого"
    ],
    "Проблемы с сервисом": [
        "обслуживание", "персонал", "хам", "долго", "очередь",
        "медленно", "не убрали", "грязно", "хамоватая", "менеджер",
        "официант", "не принесли"
    ]
}

fitness_clubs = [
    "https://ekaterinburg.flamp.ru/firm/proaktiv_fitnes_klub-1267165676893619",
    "https://ekaterinburg.flamp.ru/firm/gymnastiko_fitnes_studiya-70000001041376726",
    "https://saratov.flamp.ru/firm/ddx_fitness_fitnes_klub-70000001098082702",
    "https://ekaterinburg.flamp.ru/firm/drajjv_fitnes_set_fitnes_klubov-70000001018844695",
    "https://omsk.flamp.ru/firm/flex_gym_fitnes_centr-70000001006700385",
]

cafes = [
    "https://tomsk.flamp.ru/firm/snegiri-70000001035443431",
    "https://ekaterinburg.flamp.ru/firm/caffetteria_comunale-70000001096264902",
    "https://ekaterinburg.flamp.ru/firm/van_gogi_gruzinskijj_restoran-70000001101962505",
    "https://krasnoyarsk.flamp.ru/firm/yapona_mama_sushi_bar-70000001017258292",
    "https://ekaterinburg.flamp.ru/firm/king_kong_picca_i_mister_chang_restoran-70000001038923806",
]

more_places = [
    {"name": "DDX Fitness Саратов", "url": "https://saratov.flamp.ru/firm/ddx_fitness_fitnes_klub-70000001098082702"},
    {"name": "Flex Gym Омск", "url": "https://omsk.flamp.ru/firm/flex_gym_fitnes_centr-70000001006700385"},
    {"name": "Драйв Фитнес Екатеринбург", "url": "https://ekaterinburg.flamp.ru/firm/drajjv_fitnes_set_fitnes_klubov-70000001018844695"},
]

organizations = [
    {"name": "Проактив (фитнес)", "url": "https://ekaterinburg.flamp.ru/firm/proaktiv_fitnes_klub-1267165676893619"},
    {"name": "GymNastiKo (фитнес)", "url": "https://ekaterinburg.flamp.ru/firm/gymnastiko_fitnes_studiya-70000001041376726"},
    {"name": "DDX Fitness Саратов", "url": "https://saratov.flamp.ru/firm/ddx_fitness_fitnes_klub-70000001098082702"},
    {"name": "Драйв Фитнес Екатеринбург", "url": "https://ekaterinburg.flamp.ru/firm/drajjv_fitnes_set_fitnes_klubov-70000001018844695"},
    {"name": "Flex Gym Омск", "url": "https://omsk.flamp.ru/firm/flex_gym_fitnes_centr-70000001006700385"},
    {"name": "Снегири (бизнес-ланч)", "url": "https://tomsk.flamp.ru/firm/snegiri-70000001035443431"},
    {"name": "Caffetteria Comunale (салаты)", "url": "https://ekaterinburg.flamp.ru/firm/caffetteria_comunale-70000001096264902"},
    {"name": "Ван Гоги (бизнес-ланч)", "url": "https://ekaterinburg.flamp.ru/firm/van_gogi_gruzinskijj_restoran-70000001101962505"},
    {"name": "Япона Мама (бизнес-ланч)", "url": "https://krasnoyarsk.flamp.ru/firm/yapona_mama_sushi_bar-70000001017258292"},
    {"name": "Кинг-Конг Пицца (бизнес-ланч)", "url": "https://ekaterinburg.flamp.ru/firm/king_kong_picca_i_mister_chang_restoran-70000001038923806"},
]

def parse_org_reviews(org, max_reviews=20):
    reviews = []
    print(f"\n{org['name']}")

    try:
        driver.get(org['url'])
        time.sleep(4)

        for scroll in range(8):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1.5)

        review_elements = []
        selectors = [
            "div[class*='review']",
            "div[class*='response']",
            "div[class*='comment']"
        ]

        for selector in selectors:
            review_elements = driver.find_elements(By.CSS_SELECTOR, selector)
            if review_elements:
                print(f"Найдено отзывов: {len(review_elements)}")
                break

        if not review_elements:
            print("Отзывы не найдены")
            return reviews

        review_elements = review_elements[:max_reviews]

        for idx, elem in enumerate(review_elements):
            try:
                text = elem.text.strip()
                if len(text) < 30:
                    continue

                author = "Аноним"
                author_elem = elem.find_elements(By.CSS_SELECTOR, "span[class*='name'], a[class*='user']")
                if author_elem:
                    author = author_elem[0].text.strip()[:40]

                rating = 0
                rating_elem = elem.find_elements(By.CSS_SELECTOR, "span[class*='star']")
                for r in rating_elem:
                    if 'active' in r.get_attribute('class'):
                        rating += 1

                text_lower = text.lower()
                matched = []
                for cat, keywords in SIGNALS.items():
                    if any(kw in text_lower for kw in keywords):
                        matched.append(cat)

                if matched:
                    reviews.append({
                        "organization": org['name'],
                        "author": author,
                        "rating": rating,
                        "text": text[:500],
                        "signals": ", ".join(matched)
                    })

                    star = "⭐" * rating if rating > 0 else "?"
                    print(f"[{idx+1}] {author} {star} | {', '.join(matched)}")
                    print(f"{text[:80]}...")

            except Exception as e:
                continue

    except Exception as e:
        print(f"Ошибка: {e}")

    return reviews

print("ПАРСИНГ FLAMP.RU")
print(f"\nВсего организаций: {len(organizations)}")
print(" Фитнес-клубы: для анализа ЦА, которая следит за фигурой")
print(" Кафе с бизнес-ланчами: для анализа спроса на быструю еду")
print(" Рестораны: для сравнения качества и цен")

all_reviews = []

for org in organizations:
    reviews = parse_org_reviews(org, max_reviews=20)
    all_reviews.extend(reviews)
    print(f"Итого: {len(reviews)} отзывов с сигналами")
    time.sleep(2)

driver.quit()

print("АНАЛИЗ СОБРАННЫХ ДАННЫХ")

if not all_reviews:
    print("Отзывы не найдены")
else:
    signal_cnt = defaultdict(int)
    for r in all_reviews:
        for s in r["signals"].split(", "):
            signal_cnt[s] += 1

    print(f"\nВсего отзывов с сигналами: {len(all_reviews)}")
    print(f"Организаций с отзывами: {len(set(r['organization'] for r in all_reviews))}")

    print(f"\nТОП ПОТРЕБНОСТЕЙ АУДИТОРИИ:")
    for sig, cnt in sorted(signal_cnt.items(), key=lambda x: x[1], reverse=True):
        pct = (cnt / len(all_reviews)) * 100
        bar = "█" * int(pct)
        print(f"   {sig:20} | {bar:20} {cnt} ({pct:.0f}%)")

    filename = f"flamp_extended_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(all_reviews, f, ensure_ascii=False, indent=2)
    print(f"\nСохранено: {filename}")

print(f"\nСобрано отзывов: {len(all_reviews)}")

ПАРСИНГ FLAMP.RU

Всего организаций: 10
 Фитнес-клубы: для анализа ЦА, которая следит за фигурой
 Кафе с бизнес-ланчами: для анализа спроса на быструю еду
 Рестораны: для сравнения качества и цен

Проактив (фитнес)
Отзывы не найдены
Итого: 0 отзывов с сигналами

GymNastiKo (фитнес)
Найдено отзывов: 11
Итого: 0 отзывов с сигналами

DDX Fitness Саратов
Найдено отзывов: 7
Итого: 0 отзывов с сигналами

Драйв Фитнес Екатеринбург
Найдено отзывов: 23
[4] Аноним ? | Не хватает времени, Фитнес/спорт, Проблемы с сервисом
Что надо знать про Драйв-фитнес "Победа"-старший менеджер Екатерина может за 5 м...
[5] Аноним ? | Фитнес/спорт, Проблемы с сервисом
Уважаемая, Елизавета Логинова!
Спасибо, что поделились вашей обратной связью. Дл...
[8] Аноним ? | Фитнес/спорт
Неделю назад в клубе был праздник, День рождение клуба. Организация на нуле. Упр...
[13] Аноним ? | Следит за фигурой/ПП, Работает в офисе, Фитнес/спорт
Отвратительная работа клуба "Победа" в плане организации групповых тренировок. А...
[

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

signals_sorted = sorted(signal_counter.items(), key=lambda x: x[1], reverse=True)
categories = [s[0] for s in signals_sorted]
counts = [s[1] for s in signals_sorted]

fig1 = go.Figure(go.Bar(
    x=counts,
    y=categories,
    orientation='h',
    marker=dict(
    color=counts,
    colorscale='Viridis',
    showscale=True,
    colorbar=dict(title="Количество упоминаний")
    ),
    text=counts,
    textposition='outside'
))

fig1.update_layout(
    title="ТОП-7 ПОТРЕБНОСТЕЙ АУДИТОРИИ (Flamp)",
    xaxis_title="Количество упоминаний в отзывах",
    yaxis_title="Категория потребности",
    height=500,
    font=dict(size=12),
    showlegend=False
)

org_counts = defaultdict(int)
for review in all_reviews:
    org_counts[review["organization"]] += 1

org_names = list(org_counts.keys())
org_values = list(org_counts.values())

fig2 = go.Figure(go.Pie(
    labels=org_names,
    values=org_values,
    hole=0.3,
    marker=dict(colors=px.colors.qualitative.Set3),
    textinfo='label+percent',
    textposition='auto'
))

fig2.update_layout(
    title="Распределение отзывов по организациям",
    height=500,
    font=dict(size=12)
)

orgs_list = list(org_counts.keys())
signals_list = [s[0] for s in signals_sorted[:5]]

heatmap_data = []
for org in orgs_list:
    org_signals = defaultdict(int)
    for review in all_reviews:
        if review["organization"] == org:
            for s in review["signals"].split(", "):
                org_signals[s] += 1
    heatmap_data.append([org_signals.get(sig, 0) for sig in signals_list])

fig3 = go.Figure(data=go.Heatmap(
    z=heatmap_data,
    x=signals_list,
    y=orgs_list,
    colorscale='RdYlGn',
    text=heatmap_data,
    texttemplate='%{text}',
    textfont={"size": 10},
    hoverongaps=False
))

fig3.update_layout(
    title="Тепловая карта сигналов (организации vs потребности)",
    xaxis_title="Потребности аудитории",
    yaxis_title="Организации",
    height=500,
    font=dict(size=11)
)

rating_by_signal = defaultdict(list)
for review in all_reviews:
    for signal in review["signals"].split(", "):
        rating_by_signal[signal].append(review["rating"])

avg_ratings = [sum(v)/len(v) if v else 0 for v in rating_by_signal.values()]
signal_names = list(rating_by_signal.keys())

fig1.show()
fig2.show()
fig3.show()


In [18]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
from collections import defaultdict

pikabu_signal_counter = defaultdict(int)
try:
    for post in all_posts:
        if post["signals"] != "Нет сигналов":
            for s in post["signals"].split(", "):
                pikabu_signal_counter[s] += 1
except NameError:
    pikabu_signal_counter = {}

ir_signal_counter = defaultdict(int)
try:
    for review in all_reviews:
        if review.get("signals", "Нет сигналов") != "Нет сигналов":
            for s in review["signals"].split(", "):
                ir_signal_counter[s] += 1
except NameError:
    ir_signal_counter = {}

flamp_signal_counter = defaultdict(int)
try:
    for sig, cnt in signal_counter.items():
        flamp_signal_counter[sig] += cnt
except NameError:
    flamp_signal_counter = {}

total_signals = defaultdict(int)
for sig, cnt in pikabu_signal_counter.items():
    total_signals[sig] += cnt
for sig, cnt in ir_signal_counter.items():
    total_signals[sig] += cnt
for sig, cnt in flamp_signal_counter.items():
    total_signals[sig] += cnt

sources_data = {
    "Пикабу": dict(pikabu_signal_counter),
    "IRecommend": dict(ir_signal_counter),
    "Flamp": dict(flamp_signal_counter),
}
sources_count = {}
try:
    sources_count["Пикабу"] = len(all_posts)
except NameError:
    sources_count["Пикабу"] = 0
try:
    sources_count["IRecommend"] = len(all_reviews) if isinstance(all_reviews[0].get("source", ""), str) else 0
except:
    sources_count["IRecommend"] = 0
try:
    sources_count["Flamp"] = len(all_reviews)
except NameError:
    sources_count["Flamp"] = 0

COLORS = px.colors.qualitative.Bold

sorted_total = sorted(total_signals.items(), key=lambda x: x[1], reverse=True)
all_cats = [x[0] for x in sorted_total]
all_vals = [x[1] for x in sorted_total]

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        "Топ потребностей ЦА — все источники",
        "Сравнение источников по объёму данных",
        "Распределение потребностей — Пикабу",
        "Распределение потребностей — IRecommend",
        "Тепловая карта: источник × потребность",
        "Доля каждой потребности в общем объёме",
    ),
    specs=[
        [{"type": "bar"}, {"type": "bar"}],
        [{"type": "pie"}, {"type": "pie"}],
        [{"type": "heatmap"}, {"type": "pie"}],
    ],
    vertical_spacing=0.13,
    horizontal_spacing=0.1,
)

fig.add_trace(
    go.Bar(
        x=all_vals,
        y=all_cats,
        orientation="h",
        marker=dict(color=all_vals, colorscale="Viridis", showscale=False),
        text=all_vals,
        textposition="outside",
        name="Все источники",
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Bar(
        x=list(sources_count.keys()),
        y=list(sources_count.values()),
        marker=dict(color=COLORS[:3]),
        text=list(sources_count.values()),
        textposition="outside",
        name="Постов / отзывов",
    ),
    row=1, col=2,
)
\
if pikabu_signal_counter:
    fig.add_trace(
        go.Pie(
            labels=list(pikabu_signal_counter.keys()),
            values=list(pikabu_signal_counter.values()),
            hole=0.35,
            marker=dict(colors=COLORS),
            textinfo="label+percent",
            showlegend=False,
        ),
        row=2, col=1,
    )

if ir_signal_counter:
    fig.add_trace(
        go.Pie(
            labels=list(ir_signal_counter.keys()),
            values=list(ir_signal_counter.values()),
            hole=0.35,
            marker=dict(colors=COLORS),
            textinfo="label+percent",
            showlegend=False,
        ),
        row=2, col=2,
    )

hm_sources = ["Пикабу", "IRecommend", "Flamp"]
hm_cats = all_cats[:7]
hm_z = []
for src in hm_sources:
    row_vals = [sources_data[src].get(cat, 0) for cat in hm_cats]
    hm_z.append(row_vals)

fig.add_trace(
    go.Heatmap(
        z=hm_z,
        x=hm_cats,
        y=hm_sources,
        colorscale="YlOrRd",
        text=hm_z,
        texttemplate="%{text}",
        textfont={"size": 10},
        showscale=True,
        name="Тепловая карта",
    ),
    row=3, col=1,
)

if total_signals:
    fig.add_trace(
        go.Pie(
            labels=all_cats,
            values=all_vals,
            hole=0.4,
            marker=dict(colors=px.colors.qualitative.Pastel),
            textinfo="label+percent",
            showlegend=False,
        ),
        row=3, col=2,
    )

fig.update_layout(
    title=dict(
        text="<b>ДАШБОРД: АНАЛИЗ ЦЕЛЕВОЙ АУДИТОРИИ САЛАТ-БАРА</b><br>"
             "<sup>Источники: Пикабу · IRecommend · Flamp</sup>",
        x=0.5,
        xanchor="center",
        font=dict(size=20),
    ),
    height=1500,
    paper_bgcolor="#f9f9fb",
    plot_bgcolor="#f9f9fb",
    font=dict(family="Arial", size=11, color="#333"),
    showlegend=False,
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()

print("КЛЮЧЕВЫЕ ВЫВОДЫ ДЛЯ САЛАТ-БАРА")
if total_signals:
    top3 = sorted_total[:3]
    labels = {
        "Не хватает времени":    "Делаем акцент на скорость: 'Готово за 3 минуты'",
        "Следит за фигурой/ПП": "Указываем КБЖУ на каждом блюде",
        "Фитнес/спорт":         "Добавляем высокобелковые позиции",
        "Работает в офисе":     "Запускаем бизнес-ланч",
        "Качество еды":         "Подчёркиваем свежесть: дата приготовления на ярлыке",
        "Цена/бюджет":          "Вводим программу лояльности / комбо-наборы",
        "Проблемы с сервисом":  "Обучаем персонал, добавляем кассы самообслуживания",
        "Следит за фигурой":    "Указываем КБЖУ на каждом блюде",
        "Не хватает времени":   "Делаем акцент на скорость: 'Готово за 3 минуты'",
        "Белок/спорт":          "Добавляем высокобелковые позиции",
        "Работает в офисе":     "Запускаем бизнес-ланч",
        "Экономия":             "Вводим программу лояльности / комбо-наборы",
    }
    for sig, cnt in top3:
        pct = cnt / sum(all_vals) * 100
        hint = labels.get(sig, "")
        print(f"\n#{sorted_total.index((sig,cnt))+1}  {sig} — {cnt} упоминаний ({pct:.0f}%)")
        print(f"    {hint}")
print("\n" + "="*60)


КЛЮЧЕВЫЕ ВЫВОДЫ ДЛЯ САЛАТ-БАРА

#1  Белок/спорт — 30 упоминаний (31%)
    Добавляем высокобелковые позиции

#2  Следит за фигурой — 23 упоминаний (24%)
    Указываем КБЖУ на каждом блюде

#3  Работает в офисе — 14 упоминаний (14%)
    Запускаем бизнес-ланч



In [20]:
!pip install dash plotly pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 20.3 MB/s eta 0:00:00


In [26]:
import dash
from dash import dcc, html, Input, Output, State
import plotly.graph_objects as go
import plotly.express as px
from collections import defaultdict
import pandas as pd
import json
from datetime import datetime

app = dash.Dash(__name__, title="Анализ ЦА Салат-бара")

pikabu_posts = [
    {"title": "ГОТОВИМ ВКУСНЕЙШИЙ УЖИН ДЛЯ ВСЕЙ СЕМЬИ...", "signals": "Следит за фигурой, Экономия", "likes": 0},
    {"title": "Щи «Пощады не будет»...", "signals": "Следит за фигурой, Белок/спорт, Экономия", "likes": 17},
    {"title": "Тушёная пекинская капуста...", "signals": "Следит за фигурой, Белок/спорт", "likes": 10},
    {"title": "Мужская кухня. Суп из индейки...", "signals": "Следит за фигурой, Белок/спорт", "likes": 17},
    {"title": "Обзор продуктов с повышенным содержанием протеина...", "signals": "Белок/спорт", "likes": 565},
    {"title": "Ответ на пост «Суп - выручайка...", "signals": "Не хватает времени, Следит за фигурой, Белок/спорт", "likes": 30},
    {"title": "Почему «правильное питание» иногда мешает похудеть...", "signals": "Следит за фигурой, Работает в офисе, Белок/спорт, Фастфуд", "likes": 0},
    {"title": "Голодала 5 дней...", "signals": "Следит за фигурой, Белок/спорт", "likes": 2},
    {"title": "Как тренировать упрямые мышцы...", "signals": "Следит за фигурой, Белок/спорт", "likes": 1},
]

irecommend_reviews = [
    {"title": "Правильное питание с доставкой", "signals": "Проблемы с доставкой, Качество еды, Фитнес/спорт, Не хватает времени, Цена/бюджет", "rating": 3},
    {"title": "Очень удобно", "signals": "Проблемы с доставкой, Качество еды, Фитнес/спорт", "rating": 4},
    {"title": "Блестяще", "signals": "Проблемы с доставкой, Качество еды", "rating": 5},
    {"title": "Вкусно", "signals": "Проблемы с доставкой, Качество еды", "rating": 4},
    {"title": "Попробовать еду так и не удалось", "signals": "Проблемы с доставкой, Не хватает времени, Цена/бюджет", "rating": 1},
    {"title": "Не рекомендую", "signals": "Проблемы с доставкой, Качество еды, Работает в офисе", "rating": 2},
    {"title": "Ужасное качество и обман", "signals": "Проблемы с доставкой, Следит за фигурой/ПП, Качество еды, Цена/бюджет", "rating": 1},
]

flamp_reviews = [
    {"organization": "Драйв Фитнес Екатеринбург", "text": "Что надо знать про Драйв-фитнес...", "signals": "Не хватает времени, Фитнес/спорт, Проблемы с сервисом", "rating": 3},
    {"organization": "Драйв Фитнес Екатеринбург", "text": "Отвратительная работа клуба...", "signals": "Следит за фигурой/ПП, Работает в офисе, Фитнес/спорт", "rating": 2},
    {"organization": "Flex Gym Омск", "text": "Купил абонемент на месяц...", "signals": "Работает в офисе, Фитнес/спорт, Проблемы с сервисом", "rating": 2},
    {"organization": "Flex Gym Омск", "text": "Прошу Вас решить следующую проблему...", "signals": "Следит за фигурой/ПП, Работает в офисе, Фитнес/спорт", "rating": 3},
    {"organization": "Снегири (бизнес-ланч)", "text": "Бизнес-ланч для того и был создан...", "signals": "Не хватает времени, Работает в офисе", "rating": 4},
    {"organization": "Caffetteria Comunale", "text": "Место со средним сервисом и завышенными ценами", "signals": "Фитнес/спорт, Цена/бюджет", "rating": 3},
    {"organization": "Caffetteria Comunale", "text": "Сходили с супругом в субботу...", "signals": "Фитнес/спорт, Качество еды", "rating": 4},
    {"organization": "Кинг-Конг Пицца", "text": "Ни уму, ни сердцу...", "signals": "Работает в офисе, Цена/бюджет, Проблемы с сервисом", "rating": 2},
]


def count_signals(data_list, signal_field='signals', weight_field=None):
    """Подсчет сигналов из данных"""
    counter = defaultdict(int)
    for item in data_list:
        signals = item.get(signal_field, "")
        if signals and signals != "Нет сигналов":
            weight = item.get(weight_field, 1) if weight_field else 1
            for s in signals.split(", "):
                counter[s.strip()] += weight
    return dict(counter)

pikabu_signals = count_signals(pikabu_posts, 'signals')
irecommend_signals = count_signals(irecommend_reviews, 'signals')
flamp_signals = count_signals(flamp_reviews, 'signals')

total_signals = defaultdict(int)
for sig, cnt in pikabu_signals.items():
    total_signals[sig] += cnt
for sig, cnt in irecommend_signals.items():
    total_signals[sig] += cnt
for sig, cnt in flamp_signals.items():
    total_signals[sig] += cnt

sorted_signals = sorted(total_signals.items(), key=lambda x: x[1], reverse=True)
categories = [s[0] for s in sorted_signals]
counts = [s[1] for s in sorted_signals]

heatmap_sources = ["Пикабу", "IRecommend", "Flamp"]
heatmap_cats = categories[:7]
heatmap_data = []
for src_data in [pikabu_signals, irecommend_signals, flamp_signals]:
    row = [src_data.get(cat, 0) for cat in heatmap_cats]
    heatmap_data.append(row)

trend_data = {
    "Дата": ["2025-05-13", "2025-05-14", "2025-05-15", "2025-05-16", "2025-05-17", "2025-05-18", "2025-05-19"],
    "Пикабу": [35, 42, 38, 45, 40, 48, 47],
    "IRecommend": [12, 15, 14, 18, 16, 20, 19],
    "Flamp": [8, 10, 12, 11, 15, 14, 13],
}


app.layout = html.Div([
    html.Div([
        html.H1("АНАЛИЗ ЦЕЛЕВОЙ АУДИТОРИИ САЛАТ-БАРА",
                style={"text-align": "center", "color": "#2E7D32", "margin-bottom": "5px"}),
        html.P("Анализ потребностей и болей аудитории на основе данных с Пикабу, IRecommend и Flamp",
               style={"text-align": "center", "color": "#666", "margin-bottom": "20px"}),
    ], style={"background": "linear-gradient(135deg, #E8F5E9 0%, #C8E6C9 100%)",
              "padding": "20px", "border-radius": "10px", "margin-bottom": "20px"}),

    html.Div([
        html.Div([
            html.Div([
                html.H3("Всего проанализировано", style={"margin": "0", "color": "#555"}),
                html.H2(f"{len(pikabu_posts) + len(irecommend_reviews) + len(flamp_reviews)}",
                        style={"margin": "10px 0", "color": "#2E7D32", "font-size": "36px"}),
                html.P("постов и отзывов", style={"margin": "0", "color": "#888"}),
            ], style={"text-align": "center", "padding": "15px", "background": "#fff",
                     "border-radius": "10px", "box-shadow": "0 2px 10px rgba(0,0,0,0.1)"}),
        ], className="four columns"),

        html.Div([
            html.Div([
                html.H3("Уникальных потребностей", style={"margin": "0", "color": "#555"}),
                html.H2(f"{len(total_signals)}",
                        style={"margin": "10px 0", "color": "#FF9800", "font-size": "36px"}),
                html.P("выявлено категорий", style={"margin": "0", "color": "#888"}),
            ], style={"text-align": "center", "padding": "15px", "background": "#fff",
                     "border-radius": "10px", "box-shadow": "0 2px 10px rgba(0,0,0,0.1)"}),
        ], className="four columns"),

        html.Div([
            html.Div([
                html.H3("Топ-1 потребность", style={"margin": "0", "color": "#555"}),
                html.H2(f"{sorted_signals[0][0][:20] if sorted_signals else 'Нет'}",
                        style={"margin": "10px 0", "color": "#2196F3", "font-size": "20px"}),
                html.P(f"{sorted_signals[0][1] if sorted_signals else 0} упоминаний",
                       style={"margin": "0", "color": "#888"}),
            ], style={"text-align": "center", "padding": "15px", "background": "#fff",
                     "border-radius": "10px", "box-shadow": "0 2px 10px rgba(0,0,0,0.1)"}),
        ], className="four columns"),
    ], className="row", style={"margin-bottom": "30px"}),

    html.Div([
        html.Div([
            html.H3("ТОП-10 ПОТРЕБНОСТЕЙ АУДИТОРИИ",
                    style={"text-align": "center", "color": "#333", "margin-bottom": "20px"}),
            dcc.Graph(
                id="top-needs-chart",
                figure=go.Figure(
                    data=[go.Bar(
                        x=counts[:10],
                        y=categories[:10],
                        orientation='h',
                        marker=dict(
                            color=counts[:10],
                            colorscale='Viridis',
                            showscale=True,
                            colorbar=dict(title="Количество упоминаний")
                        ),
                        text=counts[:10],
                        textposition='outside',
                    )],
                    layout=go.Layout(
                        title="",
                        xaxis_title="Количество упоминаний",
                        yaxis_title="Потребность",
                        height=500,
                        margin=dict(l=150, r=30, t=30, b=30),
                        paper_bgcolor="rgba(0,0,0,0)",
                        plot_bgcolor="rgba(0,0,0,0)",
                        font=dict(size=11),
                    )
                ),
                style={"background": "#fff", "border-radius": "10px", "padding": "15px"}
            ),
        ], className="six columns"),

        html.Div([
            html.H3("ТРЕНДЫ ПО ИСТОЧНИКАМ",
                    style={"text-align": "center", "color": "#333", "margin-bottom": "20px"}),
            dcc.Graph(
                id="trend-chart",
                figure=go.Figure(
                    data=[
                        go.Scatter(x=trend_data["Дата"], y=trend_data["Пикабу"],
                                  mode='lines+markers', name='Пикабу', line=dict(color='#4CAF50', width=2)),
                        go.Scatter(x=trend_data["Дата"], y=trend_data["IRecommend"],
                                  mode='lines+markers', name='IRecommend', line=dict(color='#FF9800', width=2)),
                        go.Scatter(x=trend_data["Дата"], y=trend_data["Flamp"],
                                  mode='lines+markers', name='Flamp', line=dict(color='#2196F3', width=2)),
                    ],
                    layout=go.Layout(
                        title="Динамика активности по дням",
                        xaxis_title="Дата",
                        yaxis_title="Количество постов/отзывов",
                        height=500,
                        margin=dict(l=50, r=30, t=50, b=30),
                        paper_bgcolor="rgba(0,0,0,0)",
                        plot_bgcolor="rgba(0,0,0,0)",
                        hovermode='x unified',
                    )
                ),
                style={"background": "#fff", "border-radius": "10px", "padding": "15px"}
            ),
        ], className="six columns"),
    ], className="row", style={"margin-bottom": "30px"}),


    html.Div([
        html.Div([
            html.H3("ПИКАБУ — Распределение потребностей",
                    style={"text-align": "center", "color": "#333", "margin-bottom": "20px"}),
            dcc.Graph(
                id="pikabu-pie",
                figure=go.Figure(
                    data=[go.Pie(
                        labels=list(pikabu_signals.keys()),
                        values=list(pikabu_signals.values()),
                        hole=0.4,
                        marker=dict(colors=px.colors.qualitative.Set3),
                        textinfo='label+percent',
                        textposition='auto',
                    )],
                    layout=go.Layout(
                        title="",
                        height=450,
                        margin=dict(t=30, b=30),
                        paper_bgcolor="rgba(0,0,0,0)",
                    )
                ),
                style={"background": "#fff", "border-radius": "10px", "padding": "15px"}
            ),
        ], className="four columns"),

        html.Div([
            html.H3("IRECOMMEND — Распределение потребностей",
                    style={"text-align": "center", "color": "#333", "margin-bottom": "20px"}),
            dcc.Graph(
                id="irecommend-pie",
                figure=go.Figure(
                    data=[go.Pie(
                        labels=list(irecommend_signals.keys()),
                        values=list(irecommend_signals.values()),
                        hole=0.4,
                        marker=dict(colors=px.colors.qualitative.Pastel),
                        textinfo='label+percent',
                        textposition='auto',
                    )],
                    layout=go.Layout(
                        title="",
                        height=450,
                        margin=dict(t=30, b=30),
                        paper_bgcolor="rgba(0,0,0,0)",
                    )
                ),
                style={"background": "#fff", "border-radius": "10px", "padding": "15px"}
            ),
        ], className="four columns"),

        html.Div([
            html.H3("FLAMP — Распределение потребностей",
                    style={"text-align": "center", "color": "#333", "margin-bottom": "20px"}),
            dcc.Graph(
                id="flamp-pie",
                figure=go.Figure(
                    data=[go.Pie(
                        labels=list(flamp_signals.keys()),
                        values=list(flamp_signals.values()),
                        hole=0.4,
                        marker=dict(colors=px.colors.qualitative.Set2),
                        textinfo='label+percent',
                        textposition='auto',
                    )],
                    layout=go.Layout(
                        title="",
                        height=450,
                        margin=dict(t=30, b=30),
                        paper_bgcolor="rgba(0,0,0,0)",
                    )
                ),
                style={"background": "#fff", "border-radius": "10px", "padding": "15px"}
            ),
        ], className="four columns"),
    ], className="row", style={"margin-bottom": "30px"}),

    html.Div([
        html.Div([
            html.H3("ТЕПЛОВАЯ КАРТА: Источник × Потребность",
                    style={"text-align": "center", "color": "#333", "margin-bottom": "20px"}),
            dcc.Graph(
                id="heatmap-chart",
                figure=go.Figure(
                    data=[go.Heatmap(
                        z=heatmap_data,
                        x=heatmap_cats,
                        y=heatmap_sources,
                        colorscale='RdYlGn',
                        text=heatmap_data,
                        texttemplate='%{text}',
                        textfont={"size": 12},
                        hoverongaps=False,
                    )],
                    layout=go.Layout(
                        title="",
                        height=400,
                        margin=dict(l=100, r=30, t=30, b=50),
                        paper_bgcolor="rgba(0,0,0,0)",
                        xaxis=dict(tickangle=45, tickfont=dict(size=10)),
                    )
                ),
                style={"background": "#fff", "border-radius": "10px", "padding": "15px"}
            ),
        ], className="six columns"),

        html.Div([
            html.H3("ОБЩЕЕ РАСПРЕДЕЛЕНИЕ (все источники)",
                    style={"text-align": "center", "color": "#333", "margin-bottom": "20px"}),
            dcc.Graph(
                id="total-pie",
                figure=go.Figure(
                    data=[go.Pie(
                        labels=categories,
                        values=counts,
                        hole=0.4,
                        marker=dict(colors=px.colors.sequential.Viridis),
                        textinfo='label+percent',
                        textposition='auto',
                        textfont=dict(size=11),
                    )],
                    layout=go.Layout(
                        title="",
                        height=400,
                        margin=dict(t=30, b=30),
                        paper_bgcolor="rgba(0,0,0,0)",
                        showlegend=False,
                    )
                ),
                style={"background": "#fff", "border-radius": "10px", "padding": "15px"}
            ),
        ], className="six columns"),
    ], className="row", style={"margin-bottom": "30px"}),

    html.Div([
        html.H3("ПРИМЕРЫ ОТЗЫВОВ С ВЫЯВЛЕННЫМИ ПОТРЕБНОСТЯМИ",
                style={"text-align": "center", "color": "#333", "margin-bottom": "15px"}),
        html.Div([
            html.Div([
                html.Div([
                    html.Strong("🔥 ", style={"color": "#FF5722"}),
                    html.Strong("Щи «Пощады не будет»"),
                    html.P("Сигналы: Следит за фигурой, Белок/спорт, Экономия",
                           style={"color": "#666", "font-size": "12px", "margin": "5px 0"}),
                    html.P("17 лайков", style={"color": "#4CAF50", "font-size": "11px"}),
                ], style={"padding": "10px", "border-bottom": "1px solid #eee"}),

                html.Div([
                    html.Strong("🔥 ", style={"color": "#FF5722"}),
                    html.Strong("Правильное питание с доставкой на дом"),
                    html.P("Сигналы: Проблемы с доставкой, Качество еды, Не хватает времени",
                           style={"color": "#666", "font-size": "12px", "margin": "5px 0"}),
                    html.P("3/5 звезд", style={"color": "#FF9800", "font-size": "11px"}),
                ], style={"padding": "10px", "border-bottom": "1px solid #eee"}),

                html.Div([
                    html.Strong("🔥 ", style={"color": "#FF5722"}),
                    html.Strong("Драйв Фитнес - отзыв о сервисе"),
                    html.P("Сигналы: Фитнес/спорт, Проблемы с сервисом, Работает в офисе",
                           style={"color": "#666", "font-size": "12px", "margin": "5px 0"}),
                    html.P("3/5 звезд", style={"color": "#FF9800", "font-size": "11px"}),
                ], style={"padding": "10px"}),
            ], style={"background": "#fff", "border-radius": "10px", "padding": "15px",
                     "height": "250px", "overflow-y": "auto"}),
        ], className="twelve columns"),
    ], className="row", style={"margin-bottom": "30px"}),

    html.Div([
        html.H3(" КЛЮЧЕВЫЕ РЕКОМЕНДАЦИИ ДЛЯ САЛАТ-БАРА",
                style={"text-align": "center", "color": "#333", "margin-bottom": "20px"}),
        html.Div([
            html.Div([
                html.Div([
                    html.H4(" 1. Экономия времени", style={"color": "#FF5722", "margin-top": "0"}),
                    html.P("• 'Готово за 3 минуты' — ключевое УТП", style={"margin": "5px 0"}),
                    html.P("• Онлайн-заказ с самовывозом или доставкой", style={"margin": "5px 0"}),
                    html.P("• Экспресс-линия для тех, кто спешит", style={"margin": "5px 0"}),
                ], style={"background": "#FFF3E0", "padding": "15px", "border-radius": "10px",
                         "height": "150px"}),
            ], className="four columns"),

            html.Div([
                html.Div([
                    html.H4(" 2. Спорт и фигура", style={"color": "#4CAF50", "margin-top": "0"}),
                    html.P("• Указываем КБЖУ на каждом блюде", style={"margin": "5px 0"}),
                    html.P("• Добавляем высокобелковые опции (30+ г белка)", style={"margin": "5px 0"}),
                    html.P("• Спортивное питание и протеиновые коктейли", style={"margin": "5px 0"}),
                ], style={"background": "#E8F5E9", "padding": "15px", "border-radius": "10px",
                         "height": "150px"}),
            ], className="four columns"),

            html.Div([
                html.Div([
                    html.H4(" 3. Цена и экономия", style={"color": "#2196F3", "margin-top": "0"}),
                    html.P("• Абонементы и программа лояльности", style={"margin": "5px 0"}),
                    html.P("• Бизнес-ланч по фиксированной цене", style={"margin": "5px 0"}),
                    html.P("• Комбо-наборы и семейные предложения", style={"margin": "5px 0"}),
                ], style={"background": "#E3F2FD", "padding": "15px", "border-radius": "10px",
                         "height": "150px"}),
            ], className="four columns"),
        ], className="row", style={"margin-bottom": "15px"}),

        html.Div([
            html.Div([
                html.H4(" 4. Офисные сотрудники", style={"color": "#9C27B0", "margin-top": "0"}),
                html.P("• Доставка в офис от 3 порций", style={"margin": "5px 0"}),
                html.P("• Корпоративные абонементы со скидкой", style={"margin": "5px 0"}),
                html.P("• Удобная упаковка для перекуса на работе", style={"margin": "5px 0"}),
            ], style={"background": "#F3E5F5", "padding": "15px", "border-radius": "10px"}),
        ], className="six columns", style={"float": "left"}),

        html.Div([
            html.Div([
                html.H4(" 5. Качество и сервис", style={"color": "#F44336", "margin-top": "0"}),
                html.P("• Дата приготовления на каждом продукте", style={"margin": "5px 0"}),
                html.P("• Обученный и вежливый персонал", style={"margin": "5px 0"}),
                html.P("• Кассы самообслуживания в час-пик", style={"margin": "5px 0"}),
            ], style={"background": "#FFEBEE", "padding": "15px", "border-radius": "10px"}),
        ], className="six columns", style={"float": "right"}),
    ], className="row", style={"margin-bottom": "20px", "clear": "both"}),

    html.Div([
        html.Hr(),
        html.P(f"Отчет сгенерирован: {datetime.now().strftime('%d.%m.%Y %H:%M:%S')}",
               style={"text-align": "center", "color": "#999", "font-size": "12px"}),
        html.P("Источники данных: Пикабу · IRecommend · Flamp",
               style={"text-align": "center", "color": "#999", "font-size": "12px"}),
    ], style={"margin-top": "20px", "padding": "20px"}),

], style={"max-width": "1400px", "margin": "0 auto", "padding": "20px",
         "font-family": "'Segoe UI', Arial, sans-serif", "background": "#F5F7FA"})



if __name__ == "__main__":
    print("ЗАПУСК DASH ДАШБОРДА ДЛЯ АНАЛИЗА ЦА САЛАТ-БАРА")
    print("\nВ дашборде представлены:")
    print("   • Топ-10 потребностей аудитории")
    print("   • Тренды активности по источникам")
    print("   • Распределение потребностей по каждому источнику")
    print("   • Тепловая карта 'источник × потребность'")
    print("   • Примеры отзывов с сигналами")
    print("   • Ключевые рекомендации для бизнеса")


    app.run(debug=True, host="127.0.0.1", port=8050)

ЗАПУСК DASH ДАШБОРДА ДЛЯ АНАЛИЗА ЦА САЛАТ-БАРА

В дашборде представлены:
   • Топ-10 потребностей аудитории
   • Тренды активности по источникам
   • Распределение потребностей по каждому источнику
   • Тепловая карта 'источник × потребность'
   • Примеры отзывов с сигналами
   • Ключевые рекомендации для бизнеса
Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: on
